# Reference Data Generator

Generates consistent seed data used across all batch generators.
Run this notebook ONCE before running any other generator.

Creates:
  - Store reference list
  - SKU reference list
  - Shared constants used by all generators

In [0]:
import random
from datetime import date, datetime, timedelta

# ── Configuration ──────────────────────────────────────────────────────────

dbutils.widgets.text("catalog", "learning_dev", "Target Catalog")
dbutils.widgets.text("schema", "s819534", "Target Schema")
CATALOG = dbutils.widgets.get("catalog")
SCHEMA  = dbutils.widgets.get("schema")

RANDOM_SEED = 42
random.seed(RANDOM_SEED)

# ── Store Reference Data ───────────────────────────────────────────────────

STORES = [f"ST{str(i).zfill(3)}" for i in range(1, 26)]   # ST001 to ST025

REGIONS = {
    "ST001": "NORTH",  "ST002": "NORTH",  "ST003": "NORTH",
    "ST004": "NORTH",  "ST005": "NORTH",  "ST006": "SOUTH",
    "ST007": "SOUTH",  "ST008": "SOUTH",  "ST009": "SOUTH",
    "ST010": "SOUTH",  "ST011": "EAST",   "ST012": "EAST",
    "ST013": "EAST",   "ST014": "EAST",   "ST015": "EAST",
    "ST016": "WEST",   "ST017": "WEST",   "ST018": "WEST",
    "ST019": "WEST",   "ST020": "WEST",   "ST021": "CENTRAL",
    "ST022": "CENTRAL","ST023": "CENTRAL","ST024": "CENTRAL",
    "ST025": "CENTRAL"
}

TERMINALS_PER_STORE = [f"T{str(i).zfill(3)}" for i in range(1, 6)]  # T001 to T005

CASHIERS = [f"C{str(i).zfill(4)}" for i in range(1001, 1201)]  # C1001 to C1200

# ── Product Reference Data ─────────────────────────────────────────────────

# Normal SKUs: used in clean and most controlled batches
NORMAL_SKUS = [f"SKU-{str(i).zfill(5)}" for i in range(10000, 10500)]  # 500 SKUs

# New SKUs: injected only in cardinality drift batch
NEW_SKUS    = [f"SKU-{str(i).zfill(5)}" for i in range(90000, 90300)]  # 300 new SKUs

CATEGORIES  = [
    "GROCERY", "DAIRY", "BAKERY", "PRODUCE",
    "FROZEN", "BEVERAGES", "HOUSEHOLD", "PERSONAL_CARE"
]

SUPPLIERS   = [f"SUP-{str(i).zfill(3)}" for i in range(1, 51)]  # SUP-001 to SUP-050

# ── Warehouse and Location Reference ──────────────────────────────────────

WAREHOUSES     = ["WH-NORTH", "WH-SOUTH", "WH-EAST", "WH-WEST"]
LOCATION_CODES = [f"LOC-{str(i).zfill(3)}" for i in range(1, 101)]  # LOC-001 to LOC-100

# ── Date Helpers ───────────────────────────────────────────────────────────

TODAY = date.today()

def get_batch_date(offset_days: int) -> date:
    """
    Returns a date relative to today.
    Negative offset = past, positive = future.

    Historical window: offset -37 to -7  (30 days of clean baseline)
    Clean live batch:  offset -6
    Controlled batch 1 (volume drop):        offset -5
    Controlled batch 2 (null spike):         offset -4
    Controlled batch 3 (categorical):        offset -3
    Controlled batch 4 (cross-column):       offset -2
    Controlled batch 5 (cardinality):        offset -1
    Controlled batch 6 (numeric range):      offset  0  (today)
    Controlled batch 7 (combined):           offset +1  (tomorrow)
    """
    return TODAY + timedelta(days=offset_days)

HISTORICAL_DATES = [
    get_batch_date(offset)
    for offset in range(-37, -7)    # 30 days
]

CLEAN_LIVE_DATE  = get_batch_date(-6)

CONTROLLED_DATES = {
    "volume_drop"         : get_batch_date(-5),
    "null_spike"          : get_batch_date(-4),
    "categorical"         : get_batch_date(-3),
    "cross_column"        : get_batch_date(-2),
    "cardinality_drift"   : get_batch_date(-1),
    "numeric_range"       : get_batch_date(0),
    "combined"            : get_batch_date(1),
}

# ── Print Summary ──────────────────────────────────────────────────────────

print("=" * 60)
print("REFERENCE DATA SUMMARY")
print("=" * 60)
print(f"Today:                {TODAY}")
print(f"Historical window:    {HISTORICAL_DATES[0]} to {HISTORICAL_DATES[-1]}")
print(f"Clean live date:      {CLEAN_LIVE_DATE}")
print()
print("Controlled batch dates:")
for name, dt in CONTROLLED_DATES.items():
    print(f"  {name:<25} {dt}")
print()
print(f"Stores:               {len(STORES)}")
print(f"Normal SKUs:          {len(NORMAL_SKUS)}")
print(f"New SKUs (drift):     {len(NEW_SKUS)}")
print(f"Cashiers:             {len(CASHIERS)}")
print(f"Warehouses:           {len(WAREHOUSES)}")
print("=" * 60)
print("Reference data ready. Run the generator notebooks.")